# Science LLM: Google Colab Master Pipeline (T4 GPU)

GitHub Repository: [greencube-dev-boop/Science-LLM](https://github.com/greencube-dev-boop/Science-LLM)

### Google Drive Hierarchy:
- **Download Links File**: `drive/MyDrive/Science LLM/down.txt` (Optional custom PDF links)
- **PDFs and Books**: `drive/MyDrive/Science LLM/Dataset` (Target download directory)
- **Converted Clean Texts**: `drive/MyDrive/Science LLM/Dataset/extracted_texts` (High-density text corpus)
- **Checkpoints**: `drive/MyDrive/Science LLM/checkpoints`
- **Trained Model**: `drive/MyDrive/Science LLM/models/SCLLM-v<version>`
- **State & Config**: `drive/MyDrive/Science LLM/config/data.json`
- **SHA-256 Hash DB**: `drive/MyDrive/Science LLM/config/hash_db.json` (Zero duplicate downloads)

In [ ]:
# 1. Verify GPU Allocation (Requires NVIDIA Tesla T4 16GB)
!nvidia-smi

In [ ]:
# 2. Clone or Update the Science-LLM Repository in Colab
import os, sys

REPO_NAME = "Science-LLM"
if not os.path.exists(f"/content/{REPO_NAME}"):
    !git clone https://github.com/greencube-dev-boop/Science-LLM.git /content/{REPO_NAME}

%cd /content/{REPO_NAME}
!git pull

if "/content/Science-LLM" not in sys.path:
    sys.path.append("/content/Science-LLM")

In [ ]:
# 3. Install Optimized Training Dependencies
%%capture
!pip install --no-deps "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps trl peft accelerate bitsandbytes
!pip install "transformers>=4.44.0,<5.0.0" datasets pymupdf pypdf evaluate

In [ ]:
# 4. Mount Google Drive & Initialize State Tracking via config_manager
from google.colab import drive
drive.mount('/content/drive')

from config_manager import ScienceLLMConfig, create_training_args

cfg = ScienceLLMConfig(base_dir="/content/drive/MyDrive/Science LLM", version="1.0.0")

print("Google Drive Structure Verified:")
print(f"- Dataset Directory:     {cfg.dataset_dir}")
print(f"- Extracted Texts Dir:   {cfg.dataset_dir / 'extracted_texts'}")
print(f"- Checkpoints Directory: {cfg.checkpoints_dir}")
print(f"- Models Directory:      {cfg.models_dir}")
print(f"- Config & Data File:    {cfg.data_json_path}")
print(f"- SHA-256 Hash Database: {cfg.config_dir / 'hash_db.json'}")

## 5. Harvest Open Scientific Papers & Books by Tags (1,000 to 5,000 target)
Automatically seeds core textbooks (OpenStax Biology, Chemistry 2e, Anatomy, Microbiology) and queries **Europe PMC** (reviews + research) and **arXiv**:
- Tags: `organic chemistry`, `biochemistry`, `botany`, `plant biology`, `chemistry`, `biology`
- Storage-Saver Mode: Downloads to local temporary scratch space, extracts clean dehyphenated text into Google Drive `Dataset/extracted_texts/`, registers SHA-256 in `hash_db.json`, and discards temporary PDFs.
- 1,000 publications take only **~30MB of Drive space** instead of 3GB+!

In [ ]:
from science_harvester import ScienceHarvester

TAGS = [
    "organic chemistry",
    "biochemistry",
    "botany",
    "plant biology",
    "chemistry",
    "biology"
]

# Set how many papers/books you want to download (e.g. 1000 to 5000)
TARGET_PUBLICATIONS = 1000

# Set keep_raw_pdfs=True only if you have large Google Drive storage quota
harvester = ScienceHarvester(
    config=cfg,
    tags=TAGS,
    max_workers=4,
    extract_to_txt_direct=True,
    keep_raw_pdfs=False
)

downloaded_count = harvester.harvest(total_target=TARGET_PUBLICATIONS)
print(f"Harvest complete! Downloaded and processed {downloaded_count} new scientific documents.")

## 6. (Optional) Download Custom Links from `down.txt`
If you placed a `down.txt` file in `/content/drive/MyDrive/Science LLM/down.txt`, this cell downloads those custom URLs into `Dataset/`.

In [ ]:
from downloader import PDFDownloader

downloader = PDFDownloader(config=cfg)
if (cfg.base_dir / "down.txt").exists():
    print("Found down.txt! Starting batch download into Dataset/...")
    downloaded_files = downloader.process_down_txt()
    print(f"Downloaded {len(downloaded_files)} custom PDFs.")
else:
    print("No down.txt detected. Proceeding with harvested corpus.")

## 7. Batch Convert Custom PDFs to Clean Text (`PDFToTextConverter`)
If you uploaded or downloaded raw PDFs directly, this cell converts them into clean `.txt` files inside `drive/MyDrive/Science LLM/Dataset/extracted_texts/` with scientific dehyphenation and formula preservation.

In [ ]:
from pdf_converter import PDFToTextConverter

converter = PDFToTextConverter(config=cfg, max_workers=4)
converted_docs = converter.convert_all(delete_pdfs=False)
print(f"Text conversion verified. Total converted files: {converted_docs}.")

## 8. Data Tokenization & Sequence Packing for Training
Packs all converted texts into 2048-token blocks using low-RAM generator streaming.

In [ ]:
from data_pipeline import DatasetBuilder

builder = DatasetBuilder(tokenizer_name="Qwen/Qwen2.5-3B", max_seq_length=2048, config=cfg)
records = builder.process_dataset_folder()

if records > 0:
    tokenized_dataset = builder.pack_sequences_for_dapt()
    print(f"Packed {len(tokenized_dataset):,} token blocks ready for Continued Pre-Training.")
else:
    print("No documents found; using curated SciQ dataset as fallback.")

## 9. Stage 1: Continued Pre-Training (DAPT) with Auto-Resume
Adapts the base model to raw scientific nomenclature and literature.
Tracks Perplexity (PPL) in `data.json` and supports automatic resumption.

In [ ]:
from train_continued_pretrain import run_training_unsloth, run_training_standard_peft, is_unsloth_available

print(f"Can resume from previous checkpoint: {cfg.state.get('checkpoints', {}).get('can_resume', False)}")
latest_ck = cfg.get_resume_checkpoint(stage="stage1_dapt")
if latest_ck:
    print(f"Resuming from: {latest_ck}")

if is_unsloth_available():
    print("Using Unsloth fast kernels for training (r=32, alpha=64)...")
    run_training_unsloth(cfg, model_name="Qwen/Qwen2.5-3B", epochs=1, batch_size=2, grad_accum=8, lr=1e-4, lora_r=32, lora_alpha=64)
else:
    print("Using standard HuggingFace PEFT + BitsAndBytes 4-bit training (r=32, alpha=64)..." )
    run_training_standard_peft(cfg, model_name="Qwen/Qwen2.5-3B", epochs=1, batch_size=1, grad_accum=16, lr=1e-4, lora_r=32, lora_alpha=64)

## 10. Stage 2: Supervised Fine-Tuning (SFT) & Scientific Reasoning
Loads the Stage 1 adapter and fine-tunes the model on step-by-step scientific problem solving with prompt completion masking.

In [ ]:
from train_sft import train_sft

train_sft(cfg, base_model_name="Qwen/Qwen2.5-3B", epochs=2, batch_size=2, grad_accum=8, lr=2e-4)

## 11. Export Standalone Merged Model: `SCLLM-v<version>`
Merges the trained LoRA adapter into 16-bit base weights and saves directly to:
`drive/MyDrive/Science LLM/models/SCLLM-v<version>`
Synchronizes model information into `data.json`.

In [ ]:
from export_and_evaluate import merge_and_export

merge_and_export(cfg, base_model_name="Qwen/Qwen2.5-3B", version="1.0.0")
print(f"Model export complete at: {cfg.get_model_save_dir('1.0.0')}")

## 12. Benchmark Evaluation: Quantitative SciQ Accuracy & Domain Battery
Computes quantitative test accuracy on the SciQ benchmark and evaluates in-depth reasoning prompts across Organic Chemistry, Biochemistry, Botany, Plant Biology, General Chemistry, and Biology.

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from export_and_evaluate import run_quantitative_sciq_benchmark, evaluate_qualitative_prompts

model_path = str(cfg.get_model_save_dir('1.0.0'))
print(f"Loading {model_path} for evaluation...")
tokenizer = AutoTokenizer.from_pretrained(model_path)
model = AutoModelForCausalLM.from_pretrained(model_path, torch_dtype=torch.float16, device_map="auto")

# 1. Multiple-choice accuracy benchmark
acc = run_quantitative_sciq_benchmark(model, tokenizer, num_samples=50, device="cuda")
cfg.log_evaluation_result("sciq_benchmark_accuracy", score=round(acc, 2))
print(f"\n>>> Scientific Benchmark Accuracy: {acc:.2f}% <<<")

# 2. Qualitative domain prompt evaluation
evaluate_qualitative_prompts(cfg, model, tokenizer, device="cuda")
print("Evaluation complete. All results recorded in data.json.")